In [ ]:
# === 04_infer_plots.ipynb — test synthesis + variance adaptor plots ===
# Run AFTER 03_train.ipynb has produced a trained checkpoint.
#
# Settings: GPU T4 x2 (or CPU is fine for inference), Internet Off OK,
# Persistence Files only.
LANG = "Telugu"  # English run complete -> Telugu is now the active language
SSTF = "/kaggle/working/sstf"
RESTORE_STEP = 0   # 0 = latest; or set to specific step, e.g. 40000

import sys, os
os.chdir(SSTF)
sys.path.insert(0, SSTF)

In [ ]:
# 1) Run inference on the 5 test sentences (E1–E4 / T1–T4).
#    Generates wavs + per-sentence variance plots (duration, pitch, energy)
#    + a CSV with the raw values for the report.
if RESTORE_STEP:
    !python project/{LANG}/inference.py --step {RESTORE_STEP}
else:
    !python project/{LANG}/inference.py

In [ ]:
# 2) Verify the required 5 wavs + 15 plots exist
import glob, os
base = f"project/Results/{LANG}"
wavs = sorted(glob.glob(f"{base}/test_generated_audios/*.wav"))
durs = sorted(glob.glob(f"{base}/variance_adaptor_plots/duration/*.png"))
pitch = sorted(glob.glob(f"{base}/variance_adaptor_plots/pitch/*.png"))
ener = sorted(glob.glob(f"{base}/variance_adaptor_plots/energy/*.png"))
csvs = sorted(glob.glob(f"{base}/variance_adaptor_plots/*.csv"))

print(f"wavs: {len(wavs)}/5")
print(f"duration plots: {len(durs)}/5")
print(f"pitch plots: {len(pitch)}/5")
print(f"energy plots: {len(ener)}/5")
print(f"csvs: {len(csvs)}/5")

for w in wavs:
    print(" ", os.path.basename(w))

In [ ]:
# 3) Quick listen summary: show the inference summary JSON
import json
summary = f"project/Results/{LANG}/inference_summary.json"
if os.path.exists(summary):
    data = json.load(open(summary, encoding="utf-8"))
    for item in data:
        print(f"{item['id']}: pred={item['pred_sec']:.2f}s  wav={item['wav_sec']:.2f}s  f0={item['pitch_mean_hz']:.0f}Hz  energy={item['energy_mean']:.3f}")
else:
    print("inference_summary.json not found")

In [ ]:
# 4) Package everything the grader expects into a zip
#    (GitHub has 100 MB/file limit; use Git LFS for checkpoints,
#    or upload this zip to Kaggle and link it in the README.)
import zipfile, shutil
from pathlib import Path

zip_path = "/kaggle/working/sstf_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk("project/Results"):
        for f in files:
            full = Path(root) / f
            arc = str(full).replace("\\", "/")
            zf.write(full, arc)
    for root, _, files in os.walk("project/Report"):
        for f in files:
            full = Path(root) / f
            arc = str(full).replace("\\", "/")
            zf.write(full, arc)

print(f"Created {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")
# Download from Kaggle output panel, or upload to GitHub as Release asset.